In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score


In [2]:
df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA/Projets_certif/bloc_03_Machine_Learning/03_walmart/WALMART_SALES/walmart_sales.csv")

In [3]:
df = df.dropna(subset=["Date"])
df = df.dropna(subset=["Weekly_Sales"])

In [4]:
df["Store"] = df["Store"].astype(int)
store_counts = df["Store"].value_counts()
print(store_counts)

Store
3     10
14     9
13     9
19     8
1      8
18     7
7      7
5      7
6      6
4      6
8      6
2      6
12     5
17     5
20     4
16     4
9      4
15     3
10     3
11     1
Name: count, dtype: int64


In [5]:
mean = df['Unemployment'].mean()
sigma = df['Unemployment'].std()
mask = df['Unemployment'].between(mean - 3 * sigma, mean + 3 * sigma)
df = df[mask]

In [6]:
df = df[df["Store"].isin(store_counts[store_counts >= 2].index)]
print(df["Store"].value_counts())

Store
3     10
13     9
19     8
14     7
7      7
1      7
6      6
4      6
18     6
5      6
8      5
2      5
20     4
16     4
10     3
17     3
9      3
15     2
Name: count, dtype: int64


In [7]:
df['date'] = pd.to_datetime(df['Date'], format = "%d-%m-%Y")
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.dayofweek
df = df.drop(columns=['Date'])

In [8]:
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df["day"] = df["day"].astype(int)
df["weekday"] = df["weekday"].astype(int)

In [9]:
df["Holiday_Flag"] = df["Holiday_Flag"].astype("Int64")

### Split 

In [10]:
X = df.drop(columns=["Weekly_Sales"])
y = df["Weekly_Sales"]

In [11]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=X["Store"])

In [12]:
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "year", "month", "day", "weekday"]
categorical_features = ["Store", "Holiday_Flag"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()) 
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [13]:
cv = KFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
# Paramètres pour Linear Regression
params_lr = {
    "model__fit_intercept": [True, False],
}

# Paramètres pour Ridge
params_ridge = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["auto", "svd", "cholesky", "lsqr"],
    "model__fit_intercept": [True, False],
    "model__max_iter": [1000, 5000, 10000],
}

# Paramètres pour Lasso
params_lasso = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__max_iter": [1000, 5000, 10000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

# Paramètres pour ElasticNet
params_elasticnet = {
    "model__alpha": [0.01, 0.1, 1.0, 10.0],
    "model__l1_ratio": [0.1, 0.5, 0.9],
    "model__max_iter": [1000, 5000, 10000],
    "model__tol": [1e-4, 1e-3, 1e-2],
}

In [15]:
pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge()),
])

pipeline_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso()),
])

pipeline_elasticnet = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ElasticNet()),
])

In [16]:
models = {
    "Linear Regression": (pipeline_lr, params_lr),
    "Ridge": (pipeline_ridge, params_ridge),
    "Lasso": (pipeline_lasso, params_lasso),
    "ElasticNet": (pipeline_elasticnet, params_elasticnet)
}

In [17]:
scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

In [18]:
grid_results = {}

for model_name, (pipeline, params) in models.items():

    print(f"Performing Grid Search for {model_name} : ")
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=scoring,
        refit="R2",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(x_train, y_train)
    grid_results[model_name] = grid_search
    
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Best R2 score for {model_name}: {grid_search.best_score_:.4f}")
    print()

Performing Grid Search for Linear Regression : 
Fitting 5 folds for each of 2 candidates, totalling 10 fits


/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These un

Best parameters for Linear Regression: {'model__fit_intercept': True}
Best R2 score for Linear Regression: 0.7376

Performing Grid Search for Ridge : 
Fitting 5 folds for each of 120 candidates, totalling 600 fits


/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These un

Best parameters for Ridge: {'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}
Best R2 score for Ridge: 0.7871

Performing Grid Search for Lasso : 
Fitting 5 folds for each of 45 candidates, totalling 225 fits


/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.159966e+11, tolerance: 2.988e+09
  model = cd_fast.enet_coordinate_descent(
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklear

Best parameters for Lasso: {'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}
Best R2 score for Lasso: 0.7514

Performing Grid Search for ElasticNet : 
Fitting 5 folds for each of 108 candidates, totalling 540 fits


/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These un

Best parameters for ElasticNet: {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.0001}
Best R2 score for ElasticNet: 0.7824



/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/nicolasbour/miniforge3/envs/certif/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [19]:
def summarize_grid(grid, model_name, metrics):

    results = pd.DataFrame(grid.cv_results_)
    best = results.iloc[grid.best_index_]

    summary = {
        "Model": model_name,
        "Best Params": grid.best_params_
    }

    print(f"\n===== {model_name} =====")
    print(f"Meilleurs paramètres : {grid.best_params_}")

    for metric, sign in metrics.items():

        mean = best[f"mean_test_{metric}"] * sign
        std = best[f"std_test_{metric}"]

        summary[metric] = mean
        summary[f"{metric} std"] = std

        print(f"{metric:<8}: {mean:.4f} (+/- {std:.4f})")

    return (
        summary,
        results.sort_values(f"rank_test_{grid.refit}")
    )

In [20]:
metrics = {
    "R2": 1,
    "MAE": -1,
    "RMSE": -1
}

summary_rows = []
detailed_results = {}

for model_name, grid in grid_results.items():

    summary, details = summarize_grid(
        grid,
        model_name,
        metrics
    )

    summary_rows.append(summary)
    detailed_results[model_name] = details

df_grid = (
    pd.DataFrame(summary_rows)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", None)
display(df_grid.round(4))


===== Linear Regression =====
Meilleurs paramètres : {'model__fit_intercept': True}
R2      : 0.7376 (+/- 0.2463)
MAE     : 187470.3287 (+/- 84572.6644)
RMSE    : 299617.6185 (+/- 165262.9913)

===== Ridge =====
Meilleurs paramètres : {'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}
R2      : 0.7871 (+/- 0.1696)
MAE     : 183153.7099 (+/- 59031.3511)
RMSE    : 282359.0010 (+/- 118659.0583)

===== Lasso =====
Meilleurs paramètres : {'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}
R2      : 0.7514 (+/- 0.2401)
MAE     : 183597.8585 (+/- 85744.6925)
RMSE    : 288790.3143 (+/- 164852.3550)

===== ElasticNet =====
Meilleurs paramètres : {'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.0001}
R2      : 0.7824 (+/- 0.1866)
MAE     : 180593.8210 (+/- 66364.7016)
RMSE    : 280941.3629 (+/- 129826.7326)


,Model,Best Params,R2,R2 std,MAE,MAE std,RMSE,RMSE std
0,Ridge,"{'model__alpha': 0.1, 'model__fit_intercept': True, 'model__max_iter': 1000, 'model__solver': 'auto'}",0.7871,0.1696,183153.7099,59031.3511,282359.0010,118659.0583
1,ElasticNet,"{'model__alpha': 0.01, 'model__l1_ratio': 0.9, 'model__max_iter': 1000, 'model__tol': 0.0001}",0.7824,0.1866,180593.8210,66364.7016,280941.3629,129826.7326
2,Lasso,"{'model__alpha': 100.0, 'model__max_iter': 1000, 'model__tol': 0.01}",0.7514,0.2401,183597.8585,85744.6925,288790.3143,164852.3550
3,Linear Regression,{'model__fit_intercept': True},0.7376,0.2463,187470.3287,84572.6644,299617.6185,165262.9913


# Partie corrigé gpt6-luna :

# Bloc de correction proposé — cellules précédentes laissées intactes

Les cellules suivantes sont une proposition corrigée, séparée du travail original. Elles reprennent les noms de variables utilisés dans tes cellules (`df`, `X`, `y`, `x_train`, `pipeline_lr`, `cv`, `grid_results`, etc.) et détaillent chaque étape.

Ordre d’exécution :
1. Charger les données, convertir les colonnes et créer `year`, `month`, `day`, `weekday`.
2. Supprimer en pandas les lignes où une variable nécessaire est manquante/invalide, puis appliquer 3σ aux quatre colonnes exigées.
3. Ajuster et évaluer la régression linéaire; regarder ses coefficients.
4. Comparer Ridge et Lasso avec GridSearchCV.

Aucun `SimpleImputer` n’est utilisé dans cette proposition. Les lignes incomplètes sont supprimées avant le split. Les magasins rares ne sont pas supprimés; le split n’utilise pas `stratify`, car certains magasins ont trop peu d’observations.

In [26]:
# 1) Charger à nouveau le CSV pour que cette correction soit indépendante des cellules précédentes.
# Les colonnes gardent les noms de la consigne et des cellules d'origine.
df = pd.read_csv("walmart_sales.csv")

# Convertir la cible et la date. Une valeur impossible devient NaN,
# puis sera supprimée (pas d'imputation).
df["Weekly_Sales"] = pd.to_numeric(df["Weekly_Sales"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y", errors="coerce")

# Créer les quatre variables de date demandées.
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day
df["weekday"] = df["Date"].dt.dayofweek

# Date ne sera pas utilisée comme variable explicative brute.
df = df.drop(columns=["Date"])

# S'assurer que les colonnes numériques sont bien numériques.
numeric_features = [
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "year", "month", "day", "weekday",
]
for column in numeric_features:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Store"] = pd.to_numeric(df["Store"], errors="coerce")
df["Holiday_Flag"] = pd.to_numeric(df["Holiday_Flag"], errors="coerce")

print("Nombre de lignes chargées :", len(df))
display(df.head())

Nombre de lignes chargées : 150


,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,year,month,day,weekday
0,6.0,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13.0,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
2,17.0,NaN,0.0,NaN,NaN,130.719581,5.936,2012.0,7.0,27.0,4.0
3,11.0,1244390.03,0.0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN
4,6.0,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0


### Nettoyer les valeurs manquantes/invalides et appliquer le filtre 3σ

On supprime les lignes ayant une valeur manquante dans la cible ou dans une variable explicative requise. Ensuite seulement, on calcule les bornes 3σ pour `Temperature`, `Fuel_Price`, `CPI` et `Unemployment`, puis on retire toute ligne dépassant au moins une borne.

In [27]:
# Supprimer les lignes incomplètes/invalides : aucune imputation.
categorical_features = ["Store", "Holiday_Flag"]
columns_required = ["Weekly_Sales"] + numeric_features + categorical_features

rows_before_drop = len(df)
df = df.dropna(subset=columns_required).copy()
rows_removed_missing = rows_before_drop - len(df)
print("Lignes supprimées car valeur manquante/invalide :", rows_removed_missing)
print("Lignes restantes avant le filtre 3σ :", len(df))

# La consigne demande le filtre 3σ pour ces quatre variables uniquement.
outlier_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]

# Calculer la moyenne et l'écart-type sur les valeurs conservées après dropna.
means = df[outlier_features].mean()
stds = df[outlier_features].std()  # pandas utilise ddof=1 par défaut
lower_bounds = means - 3 * stds
upper_bounds = means + 3 * stds

print("Bornes 3σ par variable :")
display(pd.DataFrame({
    "Moyenne": means,
    "Écart-type": stds,
    "Borne basse": lower_bounds,
    "Borne haute": upper_bounds,
}))

# Garder une ligne seulement si les quatre mesures sont dans leurs bornes.
within_bounds = (
    df[outlier_features].ge(lower_bounds)
    & df[outlier_features].le(upper_bounds)
)
keep_rows = within_bounds.all(axis=1)
rows_removed_outliers = int((~keep_rows).sum())
df = df.loc[keep_rows].copy()

print("Lignes supprimées par le filtre 3σ :", rows_removed_outliers)
print("Lignes finales :", len(df))

# Séparer les variables explicatives X de la cible y.
X = df.drop(columns=["Weekly_Sales"])
y = df["Weekly_Sales"]

Lignes supprimées car valeur manquante/invalide : 75
Lignes restantes avant le filtre 3σ : 75
Bornes 3σ par variable :


,Moyenne,Écart-type,Borne basse,Borne haute
Temperature,60.469600,17.338982,8.452654,112.486546
Fuel_Price,3.325920,0.494012,1.843884,4.807956
CPI,176.073438,40.155442,55.607111,296.539765
Unemployment,7.660453,1.763908,2.368729,12.952178


Lignes supprimées par le filtre 3σ : 4
Lignes finales : 71


## Baseline : régression linéaire

On reprend les noms de variables de tes cellules (`X`, `y`, `x_train`, `x_test`, `y_train`, `y_test`, `numeric_features`, `categorical_features`, `preprocessor`, `pipeline_lr`). Les lignes incomplètes ont déjà été supprimées en pandas : aucun `SimpleImputer` n'est utilisé. Le pipeline ajuste le scaler et l'encodeur sur le train.

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Séparer en train/test. Sans stratify : certains magasins ont trop peu de lignes.
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Préparer les variables numériques : elles sont déjà complètes après le dropna.
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler()),
])

# Store et Holiday_Flag sont catégorielles : les encoder en colonnes indicatrices.
# Fixer les modalités observées dans X évite les catégories inconnues lors des plis CV.
store_categories = np.sort(X["Store"].unique())
holiday_categories = np.sort(X["Holiday_Flag"].unique())

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        categories=[store_categories, holiday_categories],
        drop="first",
        handle_unknown="ignore",
    )),
])

# Appliquer le bon traitement à chaque type de variable.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Baseline demandé : prétraitement + régression linéaire.
pipeline_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

# Apprendre uniquement sur le train.
pipeline_lr.fit(x_train, y_train)

# Mesurer les performances train ET test.
y_train_pred = pipeline_lr.predict(x_train)
y_test_pred = pipeline_lr.predict(x_test)

baseline_results = pd.DataFrame({
    "Train": [
        r2_score(y_train, y_train_pred),
        mean_absolute_error(y_train, y_train_pred),
        mean_squared_error(y_train, y_train_pred) ** 0.5,
    ],
    "Test": [
        r2_score(y_test, y_test_pred),
        mean_absolute_error(y_test, y_test_pred),
        mean_squared_error(y_test, y_test_pred) ** 0.5,
    ],
}, index=["R²", "MAE", "RMSE"])

display(baseline_results.round(4))

# Afficher les coefficients et leur nom après encodage.
feature_names = pipeline_lr.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": pipeline_lr.named_steps["model"].coef_,
})
coefficients["Absolute coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Absolute coefficient", ascending=False)

print("Intercept :", pipeline_lr.named_steps["model"].intercept_)
display(coefficients.head(20).round(3))

,Train,Test
R²,0.9865,0.9313
MAE,66020.2141,127770.2308
RMSE,83129.3882,149684.5746


Intercept : 986225.8173614076


,Feature,Coefficient,Absolute coefficient
10,cat__Store_4.0,2112971.996,2112971.996
18,cat__Store_13.0,1975535.937,1975535.937
16,cat__Store_10.0,1909599.094,1909599.094
11,cat__Store_5.0,-1374982.323,1374982.323
15,cat__Store_9.0,-1332161.414,1332161.414
9,cat__Store_3.0,-1310759.382,1310759.382
24,cat__Store_19.0,1310018.798,1310018.798
19,cat__Store_14.0,1259366.287,1259366.287
23,cat__Store_18.0,990475.902,990475.902
14,cat__Store_8.0,-898208.479,898208.479


### Commentaire sur le baseline

Comparer les colonnes Train et Test : un score train nettement supérieur au score test peut indiquer du surapprentissage. Le score test mesure la performance sur des lignes qui n'ont pas servi à l'ajustement, mais il dépend du split et l'échantillon est petit après suppression des lignes incomplètes. Les variables numériques étant standardisées, leurs coefficients correspondent à une variation d'un écart-type; les coefficients `Store` comparent chaque magasin à la catégorie de référence retirée par `drop="first"`. Les amplitudes entre variables numériques et indicatrices ne sont pas directement comparables et n'indiquent pas une causalité.

## Régularisation : Ridge, Lasso et GridSearchCV

On réutilise exactement le préprocesseur et les variables `x_train`, `x_test`, `y_train`, `y_test`. Aucun imputer n'est présent. `alpha_grid` contient les forces de régularisation candidates; GridSearchCV sélectionne celle qui maximise le R² moyen sur les plis du train.

In [30]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV, KFold

# Les plis sont construits seulement à partir du train.
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Tester plusieurs forces de régularisation, de faibles à fortes.
# La grille descend assez bas pour vérifier si la meilleure valeur est proche de zéro.
alpha_grid = np.logspace(-8, 2, 16)

# Scorers : sklearn maximise les scores, d'où les versions négatives de MAE/RMSE.
scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

# Standardiser la cible à l'intérieur de chaque entraînement CV.
# Les prédictions reviennent automatiquement à l'échelle des ventes.
ridge_estimator = TransformedTargetRegressor(
    regressor=Ridge(),
    transformer=StandardScaler(),
)

lasso_estimator = TransformedTargetRegressor(
    regressor=Lasso(max_iter=100_000, tol=1e-3),
    transformer=StandardScaler(),
)

# Les noms des paramètres contiennent les étapes Pipeline et TransformedTargetRegressor.
pipeline_ridge = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ridge_estimator),
])

pipeline_lasso = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lasso_estimator),
])

params_ridge = {
    "model__regressor__alpha": alpha_grid,
}

params_lasso = {
    "model__regressor__alpha": alpha_grid,
}

# Entraîner une GridSearchCV pour chaque modèle régularisé.
grid_results = {}
regularized_summary = []

models_to_search = {
    "Ridge": (pipeline_ridge, params_ridge),
    "Lasso": (pipeline_lasso, params_lasso),
}

for model_name, (pipeline, params) in models_to_search.items():
    print("Recherche des paramètres pour :", model_name)

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring=scoring,
        refit="R2",
        cv=cv,
        error_score="raise",
        n_jobs=1,
    )

    # Ne donner que le train à la recherche. Le test reste indépendant.
    grid_search.fit(x_train, y_train)
    grid_results[model_name] = grid_search

    # Évaluer le meilleur modèle sur le test une seule fois.
    best_model = grid_search.best_estimator_
    y_test_pred = best_model.predict(x_test)
    best_index = grid_search.best_index_

    regularized_summary.append({
        "Model": model_name,
        "Best alpha": grid_search.best_params_["model__regressor__alpha"],
        "CV R² moyen": grid_search.cv_results_["mean_test_R2"][best_index],
        "CV R² écart-type": grid_search.cv_results_["std_test_R2"][best_index],
        "Test R²": r2_score(y_test, y_test_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
        "Test RMSE": mean_squared_error(y_test, y_test_pred) ** 0.5,
    })

    print("Meilleurs paramètres :", grid_search.best_params_)
    print(f"Meilleur R² CV : {grid_search.best_score_:.4f}")

results_regularized = pd.DataFrame(regularized_summary)
display(results_regularized.round(4))

Recherche des paramètres pour : Ridge
Meilleurs paramètres : {'model__regressor__alpha': 0.002154434690031882}
Meilleur R² CV : 0.7521
Recherche des paramètres pour : Lasso
Meilleurs paramètres : {'model__regressor__alpha': 2.1544346900318823e-05}
Meilleur R² CV : 0.7542


,Model,Best alpha,CV R² moyen,CV R² écart-type,Test R²,Test MAE,Test RMSE
0,Ridge,0.0022,0.7521,0.1358,0.9301,126227.4524,150943.3752
1,Lasso,0.0000,0.7542,0.1765,0.9305,126836.5314,150484.3222
